# **Install dependencies**

In [0]:
%pip install --quiet confluent-kafka requests pandas

## **Load DEV secrets**

In [0]:

# ============================================================
# BANGLAPAY | PHASE 1
# KAFKA SOURCE INVENTORY
# Environment: DEV
# ============================================================

import requests
import pandas as pd

from confluent_kafka.admin import AdminClient

SCOPE = "banglapay-ingestion"
ENVIRONMENT = "dev"

# Read credentials from Databricks Secret Scope.
BOOTSTRAP_SERVERS = dbutils.secrets.get(
    SCOPE,
    "CONFLUENT_BOOTSTRAP_SERVERS"
)

KAFKA_API_KEY = dbutils.secrets.get(
    SCOPE,
    "CONFLUENT_API_KEY"
)

KAFKA_API_SECRET = dbutils.secrets.get(
    SCOPE,
    "CONFLUENT_API_SECRET"
)

SCHEMA_REGISTRY_URL = dbutils.secrets.get(
    SCOPE,
    "SCHEMA_REGISTRY_URL"
).rstrip("/")

SCHEMA_REGISTRY_KEY = dbutils.secrets.get(
    SCOPE,
    "SCHEMA_REGISTRY_API_KEY"
)

SCHEMA_REGISTRY_SECRET = dbutils.secrets.get(
    SCOPE,
    "SCHEMA_REGISTRY_API_SECRET"
)

print("Environment:", ENVIRONMENT.upper())
print("Secret loading: SUCCESS")

## **Inventory all Kafka topics**

In [0]:

# ============================================================
# KAFKA TOPIC INVENTORY
# Read-only metadata inspection.
# ============================================================

kafka_config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": KAFKA_API_KEY,
    "sasl.password": KAFKA_API_SECRET,
    "request.timeout.ms": 15000
}

admin = AdminClient(kafka_config)

metadata = admin.list_topics(timeout=20)

topic_inventory = []

for topic_name, topic_metadata in sorted(
    metadata.topics.items()
):

    if topic_name.startswith("_"):
        continue

    if topic_metadata.error is not None:
        continue

    topic_inventory.append({
        "topic": topic_name,
        "partition_count": len(
            topic_metadata.partitions
        )
    })

topic_df = pd.DataFrame(topic_inventory)

print("Accessible topics:", len(topic_inventory))

display(topic_df)

## **Inventory Schema Registry subjects**

In [0]:

# ============================================================
# CONFLUENT SCHEMA REGISTRY INVENTORY
# Read subject metadata, not schema payload contents.
# ============================================================

response = requests.get(
    f"{SCHEMA_REGISTRY_URL}/subjects",
    auth=(
        SCHEMA_REGISTRY_KEY,
        SCHEMA_REGISTRY_SECRET
    ),
    timeout=20
)

response.raise_for_status()

subject_names = response.json()

subject_inventory = []

for subject in sorted(subject_names):

    latest_response = requests.get(
        f"{SCHEMA_REGISTRY_URL}/subjects/"
        f"{requests.utils.quote(subject, safe='')}/versions/latest",

        auth=(
            SCHEMA_REGISTRY_KEY,
            SCHEMA_REGISTRY_SECRET
        ),

        timeout=20
    )

    latest_response.raise_for_status()

    item = latest_response.json()

    subject_inventory.append({
        "subject": subject,
        "version": item.get("version"),
        "schema_id": item.get("id"),
        "schema_type": item.get("schemaType") or "AVRO"
    })

subject_df = pd.DataFrame(subject_inventory)

print("Accessible subjects:", len(subject_inventory))

display(subject_df)

## **Candidate topic-to-subject mapping**

In [0]:

# ============================================================
# TOPIC AND SUBJECT MAPPING
# Detect the common TopicNameStrategy convention.
# This is a candidate mapping, not an assumption.
# ============================================================

subject_set = set(subject_names)

mapping = []

for topic in topic_df["topic"].tolist():

    key_subject = f"{topic}-key"
    value_subject = f"{topic}-value"

    mapping.append({
        "topic": topic,
        "key_subject": (
            key_subject
            if key_subject in subject_set
            else None
        ),
        "value_subject": (
            value_subject
            if value_subject in subject_set
            else None
        )
    })

mapping_df = pd.DataFrame(mapping)

display(mapping_df)

print("Inventory completed.")